# 🎬 ROTBOTS — Archive.org Scraper

---

Download and segment videos from the Internet Archive into reusable clips.

> **🤔** What happens when you mix "real" archival footage with AI-generated video?

---

In [ ]:
# SETUP
!pip install -q yt-dlp
import json, re, subprocess, shutil, time as _time
from pathlib import Path
from IPython.display import display, Markdown, Video, HTML, clear_output

from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
TEMP = Path('/content/temp_archive'); TEMP.mkdir(exist_ok=True)
print('✅ Setup complete')

In [ ]:
# SELECT SESSION
sessions = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d/'session_info.json').exists()])
if not sessions: print('❌ No sessions! Run 02_Script_Writer first.')
else:
    print('📂 Sessions:')
    for i,s in enumerate(sessions): print(f'   {i}: {s}')
SESSION_NAME = sessions[-1] if sessions else ''
SESSION_DIR = BASE_DIR / SESSION_NAME
VIDEOS_DIR = SESSION_DIR / 'videos'; VIDEOS_DIR.mkdir(exist_ok=True)
ARCHIVE_DIR = SESSION_DIR / 'archive_clips'; ARCHIVE_DIR.mkdir(exist_ok=True)
print(f'✅ Session: {SESSION_NAME}')

---
## 📥 Enter Archive.org URLs

Supported formats:
- `https://archive.org/details/IDENTIFIER`
- `https://archive.org/download/IDENTIFIER/file.mp4`
- Just the identifier: `IDENTIFIER`

**Long videos are fine!** Only the first `MAX_EXTRACT_DURATION` seconds will be used.

In [ ]:
# ============================================================
# ARCHIVE URLs — paste your links here
# ============================================================

ARCHIVE_URLS = [
    # 'https://archive.org/details/example_video',
    # 'another_archive_id',
]

# Settings
PREFER_HEIGHT = 480        # Download quality (360, 480, 720)
MAX_CLIP_DURATION = 10     # Max seconds per clip
MIN_CLIP_DURATION = 3      # Skip clips shorter than this
MAX_EXTRACT_DURATION = 180 # Only extract first N seconds (180 = 3 min). Set 0 for no limit.

print(f'🎬 {len(ARCHIVE_URLS)} archive URL(s)')
print(f'   Quality: {PREFER_HEIGHT}p, clips: {MIN_CLIP_DURATION}-{MAX_CLIP_DURATION}s')
if MAX_EXTRACT_DURATION:
    print(f'   Extract limit: first {MAX_EXTRACT_DURATION}s ({MAX_EXTRACT_DURATION//60}min) of each video')
else:
    print(f'   Extract limit: none (full video)')

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def parse_archive_id(url):
    url = url.strip().rstrip('/')
    m = re.search(r'archive\.org/(?:details|download)/([^/?#]+)', url)
    if m: return m.group(1)
    if '/' not in url and '.' not in url: return url
    raise ValueError(f'Cannot parse archive ID from: {url}')

def get_dur(path):
    try:
        r = subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration',
            '-of','default=noprint_wrappers=1:nokey=1',str(path)], capture_output=True, text=True, timeout=10)
        return float(r.stdout.strip())
    except: return 0

def download_archive_video(archive_id, dest_dir, prefer_height=480):
    """Download video from archive.org using yt-dlp."""
    url = f'https://archive.org/details/{archive_id}'
    output = str(dest_dir / f'{archive_id}.%(ext)s')
    cmd = [
        'yt-dlp', url,
        '-f', f'bestvideo[height<={prefer_height}]+bestaudio/best[height<={prefer_height}]/best',
        '--merge-output-format', 'mp4',
        '--no-playlist', '--no-warnings',
        '-o', output, '--quiet',
    ]
    try:
        subprocess.run(cmd, check=True, timeout=600)
        for f in dest_dir.iterdir():
            if f.stem == archive_id and f.suffix in ('.mp4','.mkv','.webm'): return f
    except Exception as e:
        print(f'   ❌ Download failed: {e}')
    return None

def progress_bar(current, total, label='', width=30):
    """Return an HTML progress bar string."""
    pct = current / total if total > 0 else 0
    filled = int(width * pct)
    bar = '█' * filled + '░' * (width - filled)
    return f'<div style="font-family:monospace;font-size:14px;padding:4px 0;">{bar} {current}/{total} {label} ({pct:.0%})</div>'

def segment_video_live(video_path, clip_dir, max_dur=10, min_dur=3, max_total=0):
    """Split video into clips with live progress display."""
    total = get_dur(video_path)
    if total <= 0: return []
    extract_end = min(total, max_total) if max_total > 0 else total
    est_total = int(extract_end / max_dur)
    clips = []
    t = 0; idx = 0
    t0 = _time.time()
    prog_handle = display(HTML(''), display_id=True)
    while t < extract_end:
        remaining = extract_end - t
        clip_dur = min(max_dur, remaining)
        if clip_dur < min_dur: break
        out = clip_dir / f'archive_{idx:03d}.mp4'
        cmd = ['ffmpeg','-y','-ss',str(t),'-i',str(video_path),'-t',str(clip_dur),
               '-c:v','libx264','-preset','fast','-crf','23','-an',str(out)]
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        if r.returncode == 0 and out.exists():
            clips.append({'path':str(out),'start':round(t,1),'end':round(t+clip_dur,1),'duration':round(clip_dur,1)})
            idx += 1
        t += clip_dur
        # Live progress update
        elapsed = _time.time() - t0
        speed = idx / elapsed if elapsed > 0 else 0
        eta = (est_total - idx) / speed if speed > 0 else 0
        prog_handle.update(HTML(
            f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;">' +
            f'<div style="font-size:16px;font-weight:bold;color:#4ecca3;">✂️ Segmenting: clip {idx}/{est_total}</div>' +
            progress_bar(idx, est_total, 'clips') +
            f'<div style="color:#a0a0a0;font-size:12px;margin-top:4px;">' +
            f'⏱️ {elapsed:.0f}s elapsed · {speed:.1f} clips/sec · ~{eta:.0f}s remaining · ' +
            f'📍 {t:.0f}s / {extract_end:.0f}s of video</div></div>'
        ))
    # Final state
    elapsed = _time.time() - t0
    prog_handle.update(HTML(
        f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;">' +
        f'<div style="font-size:16px;font-weight:bold;color:#4ecca3;">✅ Done! {idx} clips in {elapsed:.1f}s</div>' +
        progress_bar(idx, idx, 'clips') + '</div>'
    ))
    return clips

print('✅ Helpers loaded')

In [ ]:
# ============================================================
# DOWNLOAD AND SEGMENT
# ============================================================

all_clips = []

for i, url in enumerate(ARCHIVE_URLS):
    print(f'\n{"="*50}')
    try:
        aid = parse_archive_id(url)
    except ValueError as e:
        print(f'❌ {e}'); continue
    
    print(f'🎬 [{i+1}/{len(ARCHIVE_URLS)}] Archive ID: {aid}')
    print(f'   URL: https://archive.org/details/{aid}')
    
    # Download
    print(f'   ⬇️ Downloading ({PREFER_HEIGHT}p)...', end=' ', flush=True)
    video = download_archive_video(aid, TEMP, PREFER_HEIGHT)
    if not video:
        print('❌ Failed'); continue
    total_dur = get_dur(video)
    print(f'✅ {total_dur:.0f}s ({total_dur/60:.1f}min)')
    
    # Determine how much to extract
    if MAX_EXTRACT_DURATION > 0 and total_dur > MAX_EXTRACT_DURATION:
        extract_dur = MAX_EXTRACT_DURATION
        print(f'   ⚠️  Video is {total_dur/60:.0f}min — extracting only first {extract_dur//60}min ({extract_dur}s)')
    else:
        extract_dur = total_dur
    
    # Segment with live progress
    clip_dir = ARCHIVE_DIR / aid
    clip_dir.mkdir(exist_ok=True)
    clips = segment_video_live(video, clip_dir, MAX_CLIP_DURATION, MIN_CLIP_DURATION, MAX_EXTRACT_DURATION)
    
    for c in clips: c['archive_id'] = aid; c['source_url'] = f'https://archive.org/details/{aid}'
    all_clips.extend(clips)
    
    # Clean up full download
    video.unlink(missing_ok=True)

# Save index
if all_clips:
    with open(SESSION_DIR / 'archive_clips.json', 'w') as f:
        json.dump({'clips': all_clips, 'total': len(all_clips)}, f, indent=2)
    print(f'\n✅ {len(all_clips)} total clips saved to session')
else:
    print('\n⚠️ No clips generated. Add URLs above and re-run.')

---
## 👀 Preview Clips

In [ ]:
# PREVIEW (show first 6 clips)
if all_clips:
    display(Markdown(f'# 🎬 {len(all_clips)} Archive Clips'))
    for c in all_clips[:6]:
        display(Markdown(f'### {Path(c["path"]).name} ({c["duration"]}s)\n`{c["archive_id"]}` [{c["start"]}-{c["end"]}s]'))
        try: display(Video(c['path'], embed=True, width=480))
        except: print(f'   File: {c["path"]}')
        display(Markdown('---'))
    if len(all_clips) > 6: print(f'... and {len(all_clips)-6} more clips')
else:
    print('No clips to preview.')

---
## 🔀 Add Clips as Found Footage

In [ ]:
# Copy all archive clips to videos/ as found footage
ADD_AS_FOOTAGE = False  # Set True, then re-run this cell

if ADD_AS_FOOTAGE and all_clips:
    for i, c in enumerate(all_clips):
        shutil.copy2(Path(c['path']), VIDEOS_DIR / f'found_{i:03d}.mp4')
    print(f'✅ {len(all_clips)} clips → videos/')
    print(f'   These will be available alongside AI-generated clips in 06_Assemble.')
else:
    print('ℹ️ Set ADD_AS_FOOTAGE = True above to copy all clips to your videos/ folder.')

In [ ]:
# SUMMARY
if all_clips:
    total_dur = sum(c['duration'] for c in all_clips)
    sources = set(c['archive_id'] for c in all_clips)
    print(f'📊 Archive Summary:')
    print(f'   Sources: {len(sources)}, Clips: {len(all_clips)}, Footage: {total_dur:.0f}s ({total_dur/60:.1f}min)')

---
*ROTBOTS Workshop — LI-MA 2026*